# PHASE 0 — Project Initialization

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project"

folders = [
    "data/raw",
    "data/cleaned",
    "data/features",
    "notebooks",
    "models/trained_models",
    "models/scalers",
    "results/metrics",
    "results/plots",
    "reports",
    "app",
    "logs"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Project folder structure created at:")
print(BASE_DIR)

for root, dirs, files in os.walk(BASE_DIR):
    level = root.replace(BASE_DIR, "").count(os.sep)
    indent = " " * 4 * level
    print(f"{indent}{os.path.basename(root)}/")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder structure created at:
/content/drive/MyDrive/UP_Rainfall_Prediction_Project
UP_Rainfall_Prediction_Project/
    data/
        raw/
            nasa_power/
            era5/
            gpm_imerg/
            imd/
        cleaned/
        features/
    notebooks/
    models/
        trained_models/
        scalers/
    results/
        metrics/
        plots/
    reports/
    app/
    logs/


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project"
os.makedirs(BASE_DIR, exist_ok=True)

CONFIG_PATH = os.path.join(BASE_DIR, "config.py")

config_content = """
BASE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project"

RAW_DATA_DIR = f"{BASE_DIR}/data/raw"
CLEAN_DATA_DIR = f"{BASE_DIR}/data/cleaned"
FEATURE_DATA_DIR = f"{BASE_DIR}/data/features"

MODEL_DIR = f"{BASE_DIR}/models/trained_models"
SCALER_DIR = f"{BASE_DIR}/models/scalers"

METRICS_DIR = f"{BASE_DIR}/results/metrics"
PLOTS_DIR = f"{BASE_DIR}/results/plots"

LOG_DIR = f"{BASE_DIR}/logs"
REPORT_DIR = f"{BASE_DIR}/reports"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

RANDOM_STATE = 42
TARGET_COLUMN = "rainfall_mm"
"""

with open(CONFIG_PATH, "w") as f:
    f.write(config_content)

print("config.py successfully created at:")
print(CONFIG_PATH)


config.py successfully created at:
/content/drive/MyDrive/UP_Rainfall_Prediction_Project/config.py


# PHASE 1 — Raw Data Source Folder Setup

In [ ]:
import os

BASE_RAW_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw"

source_folders = [
    "nasa_power",
    "era5",
    "gpm_imerg",
    "imd"
]

for folder in source_folders:
    os.makedirs(os.path.join(BASE_RAW_DIR, folder), exist_ok=True)

print("Raw data source folders created:")
for folder in source_folders:
    print("-", folder)


Raw data source folders created:
- nasa_power
- era5
- gpm_imerg
- imd


In [ ]:
import requests
import pandas as pd

url = (
    "https://power.larc.nasa.gov/api/temporal/daily/point"
    "?parameters=PRECTOTCORR,RH2M,T2MDEW,PS,WS10M,WS50M,WD10M,WD50M"
    "&community=AG"
    "&longitude=80.9462"
    "&latitude=26.8467"
    "&start=20170101"
    "&end=20251130"
    "&format=JSON"
)

response = requests.get(url)
data = response.json()["properties"]["parameter"]

df = pd.DataFrame(data)
df.index.name = "date"
df.reset_index(inplace=True)

save_path = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/nasa_power/nasa_power_lucknow_raw.csv"
df.to_csv(save_path, index=False)

print("NASA POWER raw data saved at:")
print(save_path)
print("Shape:", df.shape)


NASA POWER raw data saved at:
/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/nasa_power/nasa_power_lucknow_raw.csv
Shape: (3256, 9)


In [ ]:
import requests
import pandas as pd
import os

district_coords = {
    "Lucknow": (26.8467, 80.9462),
    "Kanpur": (26.4499, 80.3319),
    "Varanasi": (25.3176, 82.9739),
    "Agra": (27.1767, 78.0081),
    "Prayagraj": (25.4358, 81.8463)
}

BASE_SAVE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/nasa_power"

for district, (lat, lon) in district_coords.items():
    url = (
        "https://power.larc.nasa.gov/api/temporal/daily/point"
        "?parameters=PRECTOTCORR,RH2M,T2MDEW,PS,WS10M,WS50M,WD10M,WD50M"
        "&community=AG"
        f"&longitude={lon}"
        f"&latitude={lat}"
        "&start=20170101"
        "&end=20251130"
        "&format=JSON"
    )

    r = requests.get(url)
    data = r.json()["properties"]["parameter"]
    df = pd.DataFrame(data)
    df.index.name = "date"
    df.reset_index(inplace=True)

    file_path = os.path.join(BASE_SAVE_DIR, f"nasa_power_{district.lower()}_raw.csv")
    df.to_csv(file_path, index=False)

    print(district, "saved | shape:", df.shape)


Lucknow saved | shape: (3256, 9)
Kanpur saved | shape: (3256, 9)
Varanasi saved | shape: (3256, 9)
Agra saved | shape: (3256, 9)
Prayagraj saved | shape: (3256, 9)


In [ ]:
import requests
import pandas as pd
import os
import time

BASE_SAVE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/nasa_power"
os.makedirs(BASE_SAVE_DIR, exist_ok=True)

up_district_coords = {
    "Agra": (27.1767, 78.0081),
    "Aligarh": (27.8974, 78.0880),
    "Ambedkar Nagar": (26.4290, 82.5400),
    "Amethi": (26.1542, 81.8140),
    "Amroha": (28.9044, 78.4673),
    "Auraiya": (26.4626, 79.5092),
    "Ayodhya": (26.7990, 82.2040),
    "Azamgarh": (26.0739, 83.1859),
    "Baghpat": (28.9445, 77.2187),
    "Bahraich": (27.5743, 81.5950),
    "Ballia": (25.7584, 84.1487),
    "Balrampur": (27.4294, 82.1807),
    "Banda": (25.4796, 80.3380),
    "Barabanki": (26.9386, 81.1834),
    "Bareilly": (28.3670, 79.4304),
    "Basti": (26.8156, 82.7634),
    "Bijnor": (29.3724, 78.1358),
    "Budaun": (28.0381, 79.1267),
    "Bulandshahr": (28.4069, 77.8498),
    "Chandauli": (25.2605, 83.2644),
    "Chitrakoot": (25.2106, 80.8591),
    "Deoria": (26.5047, 83.7796),
    "Etah": (27.5588, 78.6626),
    "Etawah": (26.7829, 79.0277),
    "Farrukhabad": (27.3905, 79.5801),
    "Fatehpur": (25.9210, 80.7996),
    "Firozabad": (27.1591, 78.3957),
    "Gautam Buddha Nagar": (28.3380, 77.5506),
    "Ghaziabad": (28.6692, 77.4538),
    "Ghazipur": (25.5855, 83.5770),
    "Gonda": (27.1330, 81.9619),
    "Gorakhpur": (26.7606, 83.3732),
    "Hamirpur": (25.9553, 80.1484),
    "Hapur": (28.7298, 77.7807),
    "Hardoi": (27.3984, 80.1311),
    "Hathras": (27.5955, 78.0521),
    "Jalaun": (26.1451, 79.3361),
    "Jaunpur": (25.7464, 82.6837),
    "Jhansi": (25.4484, 78.5685),
    "Kannauj": (27.0550, 79.9180),
    "Kanpur Nagar": (26.4499, 80.3319),
    "Kasganj": (27.8129, 78.6498),
    "Kaushambi": (25.5301, 81.3785),
    "Kushinagar": (26.7406, 83.8889),
    "Lakhimpur Kheri": (27.8973, 80.7975),
    "Lalitpur": (24.6900, 78.4192),
    "Lucknow": (26.8467, 80.9462),
    "Maharajganj": (27.1446, 83.5620),
    "Mahoba": (25.2921, 79.8724),
    "Mainpuri": (27.2285, 79.0210),
    "Mathura": (27.4924, 77.6737),
    "Mau": (25.9417, 83.5610),
    "Meerut": (28.9845, 77.7064),
    "Mirzapur": (25.1449, 82.5653),
    "Moradabad": (28.8386, 78.7733),
    "Muzaffarnagar": (29.4727, 77.7085),
    "Pilibhit": (28.6208, 79.8129),
    "Prayagraj": (25.4358, 81.8463),
    "Raebareli": (26.2309, 81.2330),
    "Rampur": (28.7973, 79.0222),
    "Saharanpur": (29.9640, 77.5460),
    "Sambhal": (28.5900, 78.5700),
    "Sant Kabir Nagar": (26.7757, 83.0710),
    "Shahjahanpur": (27.8830, 79.9120),
    "Shamli": (29.4518, 77.3175),
    "Shravasti": (27.5075, 82.0004),
    "Siddharthnagar": (27.2754, 83.1010),
    "Sitapur": (27.5619, 80.6827),
    "Sonbhadra": (24.6880, 83.0670),
    "Sultanpur": (26.2648, 82.0727),
    "Unnao": (26.5393, 80.4878),
    "Varanasi": (25.3176, 82.9739)
}

for district, (lat, lon) in up_district_coords.items():
    try:
        url = (
            "https://power.larc.nasa.gov/api/temporal/daily/point"
            "?parameters=PRECTOTCORR,RH2M,T2MDEW,PS,WS10M,WS50M,WD10M,WD50M"
            "&community=AG"
            f"&longitude={lon}"
            f"&latitude={lat}"
            "&start=20170101"
            "&end=20251130"
            "&format=JSON"
        )

        r = requests.get(url, timeout=30)
        data = r.json()["properties"]["parameter"]
        df = pd.DataFrame(data)
        df.index.name = "date"
        df.reset_index(inplace=True)

        file_path = os.path.join(
            BASE_SAVE_DIR, f"nasa_power_{district.lower().replace(' ', '_')}_raw.csv"
        )
        df.to_csv(file_path, index=False)

        print(district, "saved |", df.shape)
        time.sleep(1)

    except Exception as e:
        print("FAILED:", district, "|", e)


Agra saved | (3256, 9)
Aligarh saved | (3256, 9)
Ambedkar Nagar saved | (3256, 9)
Amethi saved | (3256, 9)
Amroha saved | (3256, 9)
Auraiya saved | (3256, 9)
Ayodhya saved | (3256, 9)
Azamgarh saved | (3256, 9)
Baghpat saved | (3256, 9)
Bahraich saved | (3256, 9)
Ballia saved | (3256, 9)
Balrampur saved | (3256, 9)
Banda saved | (3256, 9)
Barabanki saved | (3256, 9)
Bareilly saved | (3256, 9)
Basti saved | (3256, 9)
Bijnor saved | (3256, 9)
Budaun saved | (3256, 9)
Bulandshahr saved | (3256, 9)
Chandauli saved | (3256, 9)
Chitrakoot saved | (3256, 9)
Deoria saved | (3256, 9)
Etah saved | (3256, 9)
Etawah saved | (3256, 9)
Farrukhabad saved | (3256, 9)
Fatehpur saved | (3256, 9)
Firozabad saved | (3256, 9)
Gautam Buddha Nagar saved | (3256, 9)
Ghaziabad saved | (3256, 9)
Ghazipur saved | (3256, 9)
Gonda saved | (3256, 9)
Gorakhpur saved | (3256, 9)
Hamirpur saved | (3256, 9)
Hapur saved | (3256, 9)
Hardoi saved | (3256, 9)
Hathras saved | (3256, 9)
Jalaun saved | (3256, 9)
Jaunpur saved

In [ ]:
import requests
import pandas as pd
import os

SAVE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/gpm_imerg"
os.makedirs(SAVE_DIR, exist_ok=True)

cmr_url = "https://cmr.earthdata.nasa.gov/search/granules.json"

params = {
    "short_name": "GPM_3IMERGDF",
    "temporal": "2017-01-01T00:00:00Z,2025-11-30T23:59:59Z",
    "page_size": 50
}

r = requests.get(cmr_url, params=params)
entries = r.json()["feed"]["entry"]

df = pd.DataFrame([{
    "granule_id": e.get("title"),
    "start_time": e.get("time_start"),
    "end_time": e.get("time_end")
} for e in entries])

save_path = os.path.join(SAVE_DIR, "gpm_imerg_granule_list_sample.csv")
df.to_csv(save_path, index=False)

print("GPM IMERG granule metadata saved at:")
print(save_path)
print("Number of granules fetched:", len(df))


GPM IMERG granule metadata saved at:
/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/gpm_imerg/gpm_imerg_granule_list_sample.csv
Number of granules fetched: 50


In [ ]:
import os
from getpass import getpass

netrc_path = os.path.expanduser("~/.netrc")
cookies_path = os.path.expanduser("~/.urs_cookies")

username = input("Enter NASA Earthdata username: ")
password = getpass("Enter NASA Earthdata password: ")

with open(netrc_path, "w") as f:
    f.write(f"machine urs.earthdata.nasa.gov login {username} password {password}")

with open(cookies_path, "w") as f:
    f.write("")

os.chmod(netrc_path, 0o600)
os.chmod(cookies_path, 0o600)

print("Earthdata authentication configured successfully.")


Enter NASA Earthdata username: gunjan123
Enter NASA Earthdata password: ··········
Earthdata authentication configured successfully.


In [ ]:
import os
import subprocess

SAVE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/gpm_imerg"
os.makedirs(SAVE_DIR, exist_ok=True)

base_url = "https://gpm1.gesdisc.eosdis.nasa.gov/data/GPM_L3/GPM_3IMERGDF.06"
sample_dates = ["2017/01", "2017/02"]

for d in sample_dates:
    cmd = [
        "wget",
        "-nc",
        "-P", SAVE_DIR,
        "-r",
        "-np",
        "-nH",
        "--cut-dirs=6",
        f"{base_url}/{d}/"
    ]
    subprocess.run(cmd)

print("Sample GPM IMERG download completed.")


Sample GPM IMERG download completed.


In [ ]:
import os
import subprocess

SAVE_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/gpm_imerg"
os.makedirs(SAVE_DIR, exist_ok=True)

base_url = "https://gpm1.gesdisc.eosdis.nasa.gov/data/GPM_L3/GPM_3IMERGDF.06"

years = range(2017, 2026)
months = [f"{m:02d}" for m in range(1, 13)]

for y in years:
    for m in months:
        if y == 2025 and m == "12":
            break
        url = f"{base_url}/{y}/{m}/"
        cmd = [
            "wget",
            "-nc",
            "-P", SAVE_DIR,
            "-r",
            "-np",
            "-nH",
            "--cut-dirs=6",
            url
        ]
        subprocess.run(cmd)

print("Full GPM IMERG download command executed.")


Full GPM IMERG download command executed.


In [ ]:
from getpass import getpass
import os

cds_config_path = "/root/.cdsapirc"

cds_url = "https://cds.climate.copernicus.eu/api"
cds_key = getpass("Enter CDS API KEY only (no UID, no colon): ")

with open(cds_config_path, "w") as f:
    f.write(f"url: {cds_url}\n")
    f.write(f"key: {cds_key}\n")

os.chmod(cds_config_path, 0o600)

print("Updated CDS API configuration written successfully.")


Enter CDS API KEY only (no UID, no colon): ··········
Updated CDS API configuration written successfully.


In [ ]:
!pip install cdsapi


In [ ]:
import os
import cdsapi

ERA5_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/era5"
os.makedirs(ERA5_DIR, exist_ok=True)

c = cdsapi.Client()

c.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "variable": [
            "total_precipitation",
            "surface_pressure",
            "2m_temperature",
            "2m_dewpoint_temperature",
            "10m_u_component_of_wind",
            "10m_v_component_of_wind"
        ],
        "year": "2017",
        "month": "01",
        "day": "01",
        "time": "00:00",
        "area": [31, 77, 23, 85],
        "format": "netcdf"
    },
    f"{ERA5_DIR}/era5_sample_20170101.nc"
)

print("ERA5 sample NetCDF file downloaded successfully.")


2025-12-30 15:19:14,723 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
INFO:ecmwf.datastores.legacy_client:[2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-30 15:19:15,446 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entr

d8471ce60dfdb1e3c25121bc66ca476.zip:   0%|          | 0.00/97.9k [00:00<?, ?B/s]

ERA5 sample NetCDF file downloaded successfully.


In [ ]:
import os
import cdsapi

ERA5_DIR = "/content/drive/MyDrive/UP_Rainfall_Prediction_Project/data/raw/era5"
os.makedirs(ERA5_DIR, exist_ok=True)

c = cdsapi.Client()

years = [str(y) for y in range(2017, 2026)]
months = [f"{m:02d}" for m in range(1, 13)]
days = [f"{d:02d}" for d in range(1, 32)]

for year in years:
    if year == "2025":
        months_run = months[:11]
    else:
        months_run = months

    target_file = f"{ERA5_DIR}/era5_{year}.nc"

    if os.path.exists(target_file):
        print(f"{year} already downloaded, skipping.")
        continue

    c.retrieve(
        "reanalysis-era5-single-levels",
        {
            "product_type": "reanalysis",
            "variable": [
                "total_precipitation",
                "surface_pressure",
                "2m_temperature",
                "2m_dewpoint_temperature",
                "10m_u_component_of_wind",
                "10m_v_component_of_wind"
            ],
            "year": year,
            "month": months_run,
            "day": days,
            "time": ["00:00"],
            "area": [31, 77, 23, 85],
            "format": "netcdf"
        },
        target_file
    )

    print(f"ERA5 data downloaded for year {year}")

2025-12-30 15:20:44,416 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
INFO:ecmwf.datastores.legacy_client:[2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-12-30 15:20:45,141 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entr

dc31776e068c2b732da5f7443ad2f604.zip:   0%|          | 0.00/4.02M [00:00<?, ?B/s]

ERA5 data downloaded for year 2017


2025-12-30 15:21:11,190 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2025-

cc5e7b249b681be52f0e0fb50c6e2820.zip:   0%|          | 0.00/4.02M [00:00<?, ?B/s]

...

3a97c0ab641c657eb81bba4facc7896b.zip:   0%|          | 0.00/4.03M [00:00<?, ?B/s]

ERA5 data downloaded for year 2024


2025-12-30 15:24:11,506 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
INFO:ecmwf.datastores.legacy_client:[2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2025-

dc1ce33ad0a7cc020ed8c7fdfd980542.zip:   0%|          | 0.00/3.70M [00:00<?, ?B/s]

ERA5 data downloaded for year 2025
